In [1]:
import numpy as np
import matplotlib.pyplot as plt
from siriuspy.devices import S3PW
from siriuspy.search import IDSearch
from siriuspy.devices import SOFB, PowerSupplyFBP
import time
from mathphys.functions import save, load

# Define functions

In [2]:
def create_data_dict(delta, nr_ch_corrs, nr_cv_corrs, filter_jac, ref_orb):
    data = dict()
    data['delta_current'] = delta
    data['nr_ch_corrs_filter'] = nr_ch_corrs
    data['nr_cv_corrs_filter'] = nr_cv_corrs
    data['is_jacobian_filtered'] = filter_jac
    data['ref_orb'] = ref_orb
    data['timestamps'] = list()
    data['central_current_sp'] = list()
    data['initial_lateral_current_sp'] = list()
    data['central_current_rb'] = list()
    data['initial_lateral_current_rb'] = list()
    data['orb_before_jacobian'] = list()
    data['orb_after_jacobian'] = list()
    data['orbp_jacobian_lat'] = list()
    data['orbn_jacobian_lat'] = list()
    data['jacobian_lat'] = list()
    data['final_lat_current_sp'] = list()
    data['final_lat_current_rb'] = list()
    data['orbit_after_lat_current'] = list()
    data['local_corrs_current_sp'] = list()
    data['orbit_allcorrs'] = list()
    return data


def save_data(
    data,
    lat_current_rb,
    current_central,
    current_lat,
    central_current_rb,
    lateral_current_rb,
    orb_bfj,
    orb_afj,
    orbp_lat,
    orbn_lat,
    jac_lat_coils,
    final_lat_current,
    orb_after_lat,
    corr_current,
    orb_allcorrs,
):
    data['timestamps'].append(time.time())
    data['central_current_sp'].append(current_central)
    data['initial_lateral_current_sp'].append(current_lat)
    data['central_current_rb'].append(central_current_rb)
    data['initial_lateral_current_rb'].append(lateral_current_rb)
    data['orb_before_jacobian'].append(orb_bfj)
    data['orb_after_jacobian'].append(orb_afj)
    data['orbp_jacobian_lat'].append(orbp_lat)
    data['orbn_jacobian_lat'].append(orbn_lat)
    data['jacobian_lat'].append(jac_lat_coils)
    data['final_lat_current_sp'].append(final_lat_current)
    data['final_lat_current_rb'].append(lat_current_rb)
    data['orbit_after_lat_current'].append(orb_after_lat)
    data['local_corrs_current_sp'].append(corr_current)
    data['orbit_allcorrs'].append(orb_allcorrs)
    save(data, 'data_rampup.pickle', overwrite=True)


def get_sofb_orbit(sofb):
    sofb.cmd_reset()
    sofb.wait_buffer()
    orbx = sofb.orbx
    orby = sofb.orby
    orb = np.concatenate((orbx, orby))
    return orb


def set_main_coil_current(s3pw, current):
    s3pw.dev_ps_central.set_current(current, wait_mon=True)
    return


def set_lat_coils_current(s3pw, current):
    s3pw.dev_ps_lateral.set_current(current, wait_mon=True)
    return


def meas_orbdist_lat_coils(s3pw, sofb, delta=0.1):
    current = s3pw.dev_ps_lateral.current
    orb0 = get_sofb_orbit(sofb)
    s3pw.dev_ps_lateral.set_current(current - delta, wait_mon=True)
    orbn = get_sofb_orbit(sofb)
    s3pw.dev_ps_lateral.set_current(current + delta, wait_mon=True)
    orbp = get_sofb_orbit(sofb)
    s3pw.dev_ps_lateral.set_current(current, wait_mon=True)
    orbf = get_sofb_orbit(sofb)
    return orb0, orbf, orbp, orbn


def calc_jacobian_lat_coils(
    sofb, delta, dorb_jac, corr_idcs, filter_jac=False
):
    if filter_jac:
        jac = sofb.filter_local_distortions(corr_idcs, dorb_jac)
    else:
        jac = dorb_jac
    jac /= 2 * delta
    jac = np.reshape(jac, (len(jac), 1))
    return jac


def calculate_jacobian_lateral_coils(
    s3pw, sofb, delta, corr_idcs, filter_jac=False
):
    orb_bfj, orb_afj, orbp_lat, orbn_lat = meas_orbdist_lat_coils(
        s3pw, sofb, delta
    )
    jac_lat_coils = calc_jacobian_lat_coils(
        sofb, delta, orbp_lat - orbn_lat, corr_idcs, filter_jac=filter_jac
    )
    return orb_bfj, orb_afj, orbp_lat, orbn_lat, jac_lat_coils


def correct_orb_lateral_coils(
    s3pw,
    sofb,
    orb_bfj,
    ref_orb,
    jac_lat_coils,
    corr_idcs,
    max_lat_current,
    min_lat_current,
):
    dorb = orb_bfj - ref_orb
    dorb_filtered = sofb.filter_local_distortions(corr_idcs, dorb)
    delta_curr_lat = np.linalg.lstsq(
        jac_lat_coils, -1 * dorb_filtered, rcond=None
    )[0]
    final_lat_current = s3pw.dev_ps_lateral.current + delta_curr_lat
    if final_lat_current > max_lat_current:
        final_lat_current = max_lat_current
        print('Maximum value for lateral coil currents reached.')
    elif final_lat_current < min_lat_current:
        final_lat_current = min_lat_current
        print('Minimum value for lateral coil currents reached.')
    set_lat_coils_current(s3pw, final_lat_current)
    return final_lat_current


def correct_orbit_local_corrs(sofb, ref_orb, corrs, corr_idcs, respmat):
    orb_after_lat = get_sofb_orbit(sofb)
    dorb_after_lat = orb_after_lat - ref_orb
    dorb_filtered_lat = sofb.filter_local_distortions(
        corr_idcs, dorb_after_lat
    )
    delta_curr_corrs = np.linalg.lstsq(
        respmat, -1 * dorb_filtered_lat, rcond=None
    )[0]
    corr_current = np.zeros(len(corrs))
    for i, corr in enumerate(corrs):
        corr_current[i] = corr.current + delta_curr_corrs[i]
    for i, corr in enumerate(corrs):
        corr.set_current(corr_current[i], wait_mon=True)
        print(
            'Current {:.2f} A, corrector: '.format(corr.current_mon),
            corr.devname,
        )
    orb_allcorrs = get_sofb_orbit(sofb)
    return orb_after_lat, orb_allcorrs, corr_current


def procedures(current):
    """Executa os procedimentos associados a cada ponto."""
    print(f'Executing procedures at {current:.2f} A')
    time.sleep(1)


def do_all_rampup_procedures(
    current_central,
    current_lat,
    s3pw,
    sofb,
    delta,
    corr_idcs,
    filter_jac,
    ref_orb,
    max_lat_current,
    min_lat_current,
    corrs,
    respmat,
    data,
):
    # Set central currents
    set_main_coil_current(s3pw, current_central)
    # Set lateral currents
    set_lat_coils_current(s3pw, current_lat)
    central_current_rb = s3pw.dev_ps_central.current
    lateral_current_rb = s3pw.dev_ps_lateral.current
    # Calculate lateral coils jacobian
    orb_bfj, orb_afj, orbp_lat, orbn_lat, jac_lat_coils = (
        calculate_jacobian_lateral_coils(
            s3pw, sofb, delta, corr_idcs, filter_jac
        )
    )
    lat_current_rb = s3pw.dev_ps_lateral.current
    # Correct with lateral coils
    final_lat_current = correct_orb_lateral_coils(
        s3pw,
        sofb,
        orb_bfj,
        ref_orb,
        jac_lat_coils,
        corr_idcs,
        max_lat_current,
        min_lat_current,
    )
    # Correct with local correctors
    orb_after_lat, orb_allcorrs, corr_current = correct_orbit_local_corrs(
        sofb, ref_orb, corrs, corr_idcs, respmat
    )
    # Save all data
    save_data(
        data,
        lat_current_rb,
        current_central,
        current_lat,
        central_current_rb,
        lateral_current_rb,
        orb_bfj,
        orb_afj,
        orbp_lat,
        orbn_lat,
        jac_lat_coils,
        final_lat_current,
        orb_after_lat,
        corr_current,
        orb_allcorrs,
    )


def ramp_s3pw_automatic(
    s3pw,
    sofb,
    delta,
    corr_idcs,
    filter_jac,
    ref_orb,
    max_lat_current,
    min_lat_current,
    corrs,
    respmat,
    data,
    main_currents,
    lateral_currents
):

    automatic = False
    auto_target = None

    for i, current in enumerate(main_currents):
        # Modo assistido
        if not automatic:
            print(f'\nCurrent point: {current:.2f} A')

            while True:
                command = (
                    input(
                        '[ENTER] Next step | [a] Automatic mode | [q] Abort: '
                    )
                    .strip()
                    .lower()
                )

                if command == '':
                    break

                elif command == 'a':
                    while True:
                        value = input('Automatic mode up to current [A]: ')
                        try:
                            auto_target = float(value)
                        except ValueError:
                            print(
                                'Invalid input. Please enter a numeric value.'
                            )
                            continue

                        if auto_target < current:
                            print('Target must be greater than current.')
                            continue

                        automatic = True
                        break
                    break

                elif command == 'q':
                    print('Ramp aborted.')
                    return

                else:
                    print('Invalid command.')

        # Verifica se já ultrapassou o limite automático
        if automatic and current > auto_target:
            print('Automatic target reached.')
            break

        # Executa o ponto
        current_central = current
        current_lat = lateral_currents[i]
        do_all_rampup_procedures(
            current_central,
            current_lat,
            s3pw,
            sofb,
            delta,
            corr_idcs,
            filter_jac,
            ref_orb,
            max_lat_current,
            min_lat_current,
            corrs,
            respmat,
            data,
        )

    print('Ramp finished.')

# Create and configure devices

In [3]:
# Create S3PW device
s3pw = S3PW(S3PW.DEVICES.S3PW_15SP)

In [ ]:
# Create and configure SOFB
sofb = SOFB(SOFB.DEVICES.SI)
sofb.cmd_turn_off_autocorr()
sofb.cmd_change_opmode_to_sloworb()

In [4]:
# Get correctors devices
iddevname = IDSearch.conv_beamline_2_idname('SUSSUARANA')
chnames = IDSearch.conv_idname_2_idff_chnames(iddevname)
cvnames = IDSearch.conv_idname_2_idff_cvnames(iddevname)
ch1 = PowerSupplyFBP(chnames[0])
ch2 = PowerSupplyFBP(chnames[1])
cv1 = PowerSupplyFBP(cvnames[0])
cv2 = PowerSupplyFBP(cvnames[1])
corrs = [ch1, ch2, cv1, cv2]

In [12]:
data = np.loadtxt('Currents_ramp-up.txt', skiprows=1)
ic_sw = data[:, 0]
il_sw = data[:, 1]
main_currents = np.arange(2, 236, 1)
main_currents = np.concatenate(([1.4], main_currents))
il_interp = np.interp(main_currents, ic_sw, il_sw)
idcs = il_interp < 1.5
il_interp[idcs] = 1.5

In [ ]:
# Change opmode to slow_ref
s3pw.dev_ps_central.opmode = 0
s3pw.dev_ps_lateral.opmode = 0
ch1.opmode = 0
ch2.opmode = 0
cv1.opmode = 0
cv2.opmode = 0

# Meas local correctors respmat

In [ ]:
# Turn on all corrs
for i, corr in enumerate(corrs):
    corr.cmd_turn_on()
    corr.set_current(0, wait_mon=True)
    print(
        'Current {:.2f} A, corrector: '.format(corr.current_mon), corr.devname
    )

delta = 1  # [A]
respmat = np.zeros((320, len(corrs)))
sofb.correct_orbit_manually(nr_iters=5, residue=0.5)
for i, corr in enumerate(corrs):
    corr.set_current(delta, wait_mon=True)
    print(
        'Current {:.2f} A, corrector: '.format(corr.current_mon), corr.devname
    )
    orbp = get_sofb_orbit(sofb)
    corr.set_current(-1*delta, wait_mon=True)
    print(
        'Current {:.2f} A, corrector: '.format(corr.current_mon), corr.devname
    )
    orbn = get_sofb_orbit(sofb)
    corr.set_current(0, wait_mon=True)
    print(
        'Current {:.2f} A, corrector: '.format(corr.current_mon), corr.devname
    )
    dorb = orbp - orbn
    respmat[:, i] = dorb/(2*delta)

save(respmat, 'respmat_local_corres.pickle')

# Ramp S3PW

## Test single point

In [ ]:
# Parameters
max_lat_current = 254  # [A]
min_lat_current = 1.4  # [A]
filter_jac = False
ss = 15
nr_ch_corrs = 4
nr_cv_corrs = 4
delta = 0.1  # [A]
corr_idcs = sofb.find_closest_corrs_ssection(15, nr_ch_corrs, nr_cv_corrs)
sofb.correct_orbit_manually(nr_iters=5, residue=0.5)
ref_orb = get_sofb_orbit(sofb)

# Create data dict
data = create_data_dict(delta, nr_ch_corrs, nr_cv_corrs, filter_jac, ref_orb)

current_central = main_currents[0]
current_lat = il_interp[0]

In [ ]:
# Set central currents
set_main_coil_current(s3pw, current_central)

In [ ]:
# Set lateral currents
set_lat_coils_current(s3pw, current_lat)
central_current_rb = s3pw.dev_ps_central.current
lateral_current_rb = s3pw.dev_ps_lateral.current

In [ ]:
# Calculate lateral coils jacobian
orb_bfj, orb_afj, orbp_lat, orbn_lat, jac_lat_coils = (
    calculate_jacobian_lateral_coils(s3pw, sofb, delta, corr_idcs, filter_jac)
)
lat_current_rb = s3pw.dev_ps_lateral.current

In [ ]:
# Correct with lateral coils
final_lat_current = correct_orb_lateral_coils(
    s3pw,
    sofb,
    orb_bfj,
    ref_orb,
    jac_lat_coils,
    corr_idcs,
    max_lat_current,
    min_lat_current,
)

In [ ]:
# Correct with local correctors
orb_after_lat, orb_allcorrs, corr_current = correct_orbit_local_corrs(
    sofb, ref_orb, corrs, corr_idcs, respmat
)

In [ ]:
# Save all data
save_data(
    data,
    lat_current_rb,
    current_central,
    current_lat,
    central_current_rb,
    lateral_current_rb,
    orb_bfj,
    orb_afj,
    orbp_lat,
    orbn_lat,
    jac_lat_coils,
    final_lat_current,
    orb_after_lat,
    corr_current,
    orb_allcorrs,
)

## Automatic ramp-up

In [57]:
# Parameters
max_lat_current = 254  # [A]
min_lat_current = 1.4  # [A]
filter_jac = False
ss = 15
nr_ch_corrs = 4
nr_cv_corrs = 4
delta = 0.1  # [A]
corr_idcs = sofb.find_closest_corrs_ssection(15, nr_ch_corrs, nr_cv_corrs)
sofb.correct_orbit_manually(nr_iters=5, residue=0.5)
ref_orb = get_sofb_orbit(sofb)

# Create data dict
data = create_data_dict(delta, nr_ch_corrs, nr_cv_corrs, filter_jac, ref_orb)

ramp_s3pw_automatic(
    s3pw,
    sofb,
    delta,
    corr_idcs,
    filter_jac,
    ref_orb,
    max_lat_current,
    min_lat_current,
    corrs,
    respmat,
    data,
    main_currents,
    il_interp
)



Current point: 1.40 A
Setting current: 1.40 A
Executing procedures at 1.40 A

Current point: 2.00 A
Setting current: 2.00 A
Executing procedures at 2.00 A
Setting current: 3.00 A
Executing procedures at 3.00 A
Setting current: 4.00 A
Executing procedures at 4.00 A
Setting current: 5.00 A
Executing procedures at 5.00 A
Setting current: 6.00 A
Executing procedures at 6.00 A
Setting current: 7.00 A
Executing procedures at 7.00 A
Setting current: 8.00 A
Executing procedures at 8.00 A
Setting current: 9.00 A
Executing procedures at 9.00 A
Setting current: 10.00 A
Executing procedures at 10.00 A
Automatic target reached.
Ramp finished.
